# Тема 1. Анализ данных с Pandas

## Практика: пассажиры «Титаника»

На теории мы нашли первое правило, предсказывающее отток телеком-клиентов, — 85.8% точности «на коленке», без единой модели. Сейчас тот же набор приёмов (`value_counts`, `crosstab`, `groupby`, булева индексация) применим к другому классическому датасету — и найдём, что определяло выживание на «Титанике».

Самостоятельная работа. В ячейках с комментарием `# Ваш код здесь` — напишите код и получите ответ. Под каждым заданием — варианты ответа, сверьтесь с ними (в реальном отчёте о работе можно просто указать букву верного варианта).

Если где-то не хватает метода из теории — загляните в `lesson01_pandas_theory.ipynb`.

**Это файл с решениями — используйте его для самопроверки, а не вместо самостоятельной работы.**

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 2)

**Загрузим данные.**

In [2]:
data = pd.read_csv("../data/titanic_train.csv", index_col="PassengerId")
data.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.28,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.92,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.10,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,NaN,S


In [3]:
data.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,714.00,891.00,891.00,891.00
mean,0.38,2.31,29.70,0.52,0.38,32.20
std,0.49,0.84,14.53,1.10,0.81,49.69
min,0.00,1.00,0.42,0.00,0.00,0.00
25%,0.00,2.00,20.12,0.00,0.00,7.91
50%,0.00,3.00,28.00,0.00,0.00,14.45
75%,1.00,3.00,38.00,1.00,0.00,31.00
max,1.00,3.00,80.00,8.00,6.00,512.33


**Пример: создание нового признака.**

Часто в анализе полезно сгруппировать непрерывный признак в категории. Сделаем возрастную категорию из `Age` двумя способами — сначала циклом, потом через `apply` (второй способ будет часто использоваться дальше по курсу).

**Важно:** в `Age` есть пропуски (проверьте — `data['Age'].isna().sum()`, увидите 177). Если не обработать их отдельно, `age < 30` и `age < 55` для `NaN` дают `False`, и все пассажиры с неизвестным возрастом молча попадут в категорию "55+" — это тихая ошибка, её легко не заметить. Обрабатываем пропуск явно.

In [4]:
def age_category(age):
    """
    NaN -> NaN (возраст неизвестен, не категоризируем)
    < 30 -> 1
    30 <= age < 55 -> 2
    >= 55 -> 3
    """
    if pd.isna(age):
        return np.nan
    if age < 30:
        return 1
    elif age < 55:
        return 2
    else:
        return 3

In [5]:
age_categories = [age_category(age) for age in data["Age"]]
data["Age_category"] = age_categories

In [6]:
data["Age_category"] = data["Age"].apply(age_category)

Проверим, что пропуски действительно остались пропусками, а не утекли в одну из категорий:

In [7]:
data["Age_category"].isna().sum()

np.int64(177)

**1. Сколько мужчин / женщин находилось на борту?**
- 412 мужчин и 479 женщин
- 314 мужчин и 577 женщин
- 479 мужчин и 412 женщин
- **<font color='green'>577 мужчин и 314 женщин [+]</font>**

In [8]:
data["Sex"].value_counts()

Sex
male      577
female    314
Name: count, dtype: int64

**2. Выведите распределение переменной `Pclass` (социально-экономический статус), а также это же распределение отдельно для мужчин и женщин. Сколько было мужчин 2-го класса?**
- 104
- **<font color='green'>108 [+]</font>**
- 112
- 125

In [9]:
pd.crosstab(data["Pclass"], data["Sex"], margins=True)

Sex,female,male,All
Pclass,,,
1,94,122,216
2,76,108,184
3,144,347,491
All,314,577,891


**3. Каковы медиана и стандартное отклонение платежей (`Fare`)? Округлите до 2 знаков.**
- **<font color='green'>Медиана – 14.45, стандартное отклонение – 49.69 [+]</font>**
- Медиана – 15.1, стандартное отклонение – 12.15
- Медиана – 13.15, стандартное отклонение – 35.3
- Медиана – 17.43, стандартное отклонение – 39.1

In [10]:
round(data["Fare"].median(), 2), round(data["Fare"].std(), 2)

(np.float64(14.45), np.float64(49.69))

**4. Правда ли, что средний возраст выживших выше, чем у погибших?**
- Да
- **<font color='green'>Нет [+]</font>**

In [11]:
data.groupby("Survived")["Age"].mean()

Survived
0    30.63
1    28.34
Name: Age, dtype: float64

**5. Правда ли, что люди моложе 30 лет выживали чаще, чем люди старше 60 лет? Каковы доли выживших в обеих группах?**
- 22.7% среди молодых и 40.6% среди старых
- **<font color='green'>40.6% среди молодых и 22.7% среди старых [+]</font>**
- 35.3% среди молодых и 27.4% среди старых
- 27.4% среди молодых и 35.3% среди старых

In [12]:
young_survived = data.loc[data["Age"] < 30, "Survived"]
old_survived = data.loc[data["Age"] > 60, "Survived"]
round(100 * young_survived.mean(), 1), round(100 * old_survived.mean(), 1)

(np.float64(40.6), np.float64(22.7))

**6. Правда ли, что женщины выживали чаще мужчин? Каковы доли выживших в обеих группах?**
- 30.2% среди мужчин и 46.2% среди женщин
- 35.7% среди мужчин и 74.2% среди женщин
- 21.1% среди мужчин и 46.2% среди женщин
- **<font color='green'>18.9% среди мужчин и 74.2% среди женщин [+]</font>**

In [13]:
male_survived = data[data["Sex"] == "male"]["Survived"]
female_survived = data[data["Sex"] == "female"]["Survived"]
round(100 * male_survived.mean(), 1), round(100 * female_survived.mean(), 1)

(np.float64(18.9), np.float64(74.2))

**7. Найдите самое популярное имя среди пассажиров-мужчин.**
- Charles
- Thomas
- **<font color='green'>William [+]</font>**
- John

In [14]:
first_names = data.loc[data["Sex"] == "male", "Name"].apply(
    lambda full_name: full_name.split(",")[1].split()[1]
)
first_names.value_counts().head()

Name
William    35
John       25
George     14
Charles    13
Thomas     13
Name: count, dtype: int64

**8. Как средний возраст мужчин / женщин зависит от класса обслуживания (`Pclass`)? Выберите все верные утверждения:**
- **<font color='green'>В среднем мужчины 1-го класса старше 40 лет [+]</font>**
- В среднем женщины 1-го класса старше 40 лет
- **<font color='green'>Мужчины всех классов в среднем старше женщин того же класса [+]</font>**
- **<font color='green'>В среднем пассажиры 1-го класса старше пассажиров 2-го, а те — старше пассажиров 3-го [+]</font>**

In [15]:
pd.crosstab(data["Pclass"], data["Sex"], values=data["Age"], aggfunc="mean")

Sex,female,male
Pclass,,
1,34.61,41.28
2,28.72,30.74
3,21.75,26.51


**9. Как доля выживших зависит от `Age_category`, которую мы создали выше (1 = моложе 30, 2 = 30–54, 3 = 55+)? Есть ли монотонный тренд «чем старше, тем реже выживали»?**
- Да, доля выживших монотонно падает с возрастом
- **<font color='green'>Нет, у категории 2 (30–54) доля выживших даже немного выше, чем у категории 1 [+]</font>**
- Нет, все три категории выживали одинаково часто
- Нельзя сказать — данных недостаточно

In [16]:
data.groupby("Age_category")["Survived"].mean()

Age_category
1.0    0.41
2.0    0.42
3.0    0.31
Name: Survived, dtype: float64

---
### Задача со звёздочкой (доп.)

**10. Постройте `pivot_table`: средняя `Fare` по `Pclass` (строки) и `Embarked` (столбцы). В каком порту посадки пассажиры 1-го класса в среднем платили больше всего?**

Открытый вопрос, вариантов ответа нет — сверьтесь с решением.

In [17]:
data.pivot_table(values="Fare", index="Pclass", columns="Embarked", aggfunc="mean")

Embarked,C,Q,S
Pclass,,,
1,104.72,90.00,70.36
2,25.36,12.35,20.33
3,11.21,11.18,14.64


---
## Итог

Дальше в курсе — визуализация (следующее занятие), а после неё первые модели машинного обучения: baseline из прошлого ноутбука (правило по двум признакам) станет ориентиром, с которым будем сравнивать качество моделей.